In [1]:
import pandas as pd
import numpy as np
import re # Regular expression library

print("--- Phase 2 (REVISED): Text Preprocessing and Enhanced Feature Engineering ---")

try:
    # 1. Load the dataset from Phase 1
    df = pd.read_csv("MASTER_combined_and_cleaned.csv")
    print(f"Successfully loaded 'MASTER_combined_and_cleaned.csv'. Shape: {df.shape}")

    # 2. Prepare the text column
    text_col = df['text_to_analyze'].str.lower().fillna('')
    print("Prepared 'text_to_analyze' column (lowercase, fillna).")

    # 3. Define ENHANCED Lexicons (as regex patterns)
    # Based on your expert input!
    
    # --- Gender Identity Lexicons ---
    lexicon_gender_inclusive = {
        'count_menstruator': r'menstruator(s)?',
        'count_person_bleed': r'person who bleed(s)?|people who bleed',
        'count_person_uterus': r'person with a uterus|people with a uterus'
    }

    lexicon_gender_exclusive = {
        'count_woman': r'\bwoman|\bwomen',
        'count_female': r'\bfemale(s)?',
        'count_lady': r'\blady|\bladies'
    }

    lexicon_gender_identity = {
        'count_trans': r'\btrans\b',
        'count_transgender': r'transgender',
        'count_nonbinary': r'non-binary|nonbinary|\bnb\b|\benby\b',
        'count_ftm': r'\bftm\b',
        'count_trans_man': r'trans ?man|trans ?men',
        'count_transmasc': r'transmasc',
        'count_afab': r'\bafab\b'
    }

    # --- Product Lexicons ---
    lexicon_product_traditional = {
        'count_pad': r'pad(s)?|sanitary napkin(s)?|liner(s)?',
        'count_tampon': r'tampon(s)?'
    }

    lexicon_product_alternative = {
        'count_cup': r'menstrual cup(s)?|diva ?cup(s)?',
        'count_underwear': r'period underwear|period pant(y|ies)',
        'count_reusable_pad': r'reusable pad(s)?|cloth pad(s)?',
        'count_disc': r'menstrual disc(s)?'
    }

    # --- NEW: Product Stigma/Fear Narrative (from your input) ---
    lexicon_narrative_product_stigma = {
        'count_stigma_stuck': r'get stuck|getting stuck|stuck inside',
        'count_stigma_virginity': r'virginity|hymen|\bvirgin\b',
        'count_stigma_pain': r'\bhurt(s)?|\bhard to insert|\bhard to remove|painful|problematic|\btense|\btensing up|\bhard to get out',
        'count_stigma_general': r'stigma(tized)?|shame|ashamed|embarrass(ed|ing)?|gross|disgusting|dirty|unhygienic|taboo'
    }

    # --- NEW: Product Positive/Empowerment Narrative (from your input) ---
    lexicon_narrative_product_positive = {
        'count_positive_freedom': r'swim|dance|swimming',
        'count_positive_comfort': r'no rashes|no leakage|doesn(t)? feel|can(t)? feel it|comfortable',
        'count_positive_general': r'best part|love (it|my cup)|life changing|empowerment|game changer|sorted'
    }
    
    # --- NEW: Identity Stigma/Exclusion Narrative (from your input) ---
    lexicon_narrative_identity_stigma = {
        'count_dysphoria': r'dysphoria|dysphoric',
        'count_exclusion_places': r'male washroom|men(s)? washroom|male bathroom|men(s)? bathroom',
        'count_exclusion_media': r'woman centric|women centric|symbol of womanhood|pink|floral|ads|advertisement(s)?'
    }

    # --- Other Key Narratives ---
    lexicon_narrative_sustainability = {
        'count_sustainability': r'sustainable|sustainability',
        'count_eco_friendly': r'eco-friendly|eco friendly',
        'count_environment': r'environment(al)?',
        'count_waste': r'\bwaste\b|landfill(s)?',
        'count_reusable': r'reusable'
    }
    
    lexicon_narrative_health = {
        'count_health_safe': r'\bhealth|\bsafe|\bsafety',
        'count_tss': r'toxic shock|tss',
        'count_infection': r'infection(s)?',
        'count_chemical': r'chemical(s)?|bleach|dioxin(s)?',
        'count_doctor': r'doctor(s)?|obgyn|gynecologist'
    }
    
    lexicon_narrative_cost = {
        'count_cost': r'cost(s)?|price|expensive|affordable',
        'count_save_money': r'save money|saving money',
        'count_cheap': r'cheap(er)?'
    }

    # Combine all lexicons into one list for iteration
    all_lexicons = [
        lexicon_gender_inclusive, lexicon_gender_exclusive, lexicon_gender_identity,
        lexicon_product_traditional, lexicon_product_alternative,
        lexicon_narrative_product_stigma, lexicon_narrative_product_positive,
        lexicon_narrative_identity_stigma, lexicon_narrative_sustainability,
        lexicon_narrative_health, lexicon_narrative_cost
    ]

    # 4. Feature Engineering: Count Occurrences
    print("\nStarting feature engineering (counting ENHANCED lexicon occurrences)...")
    for lexicon_group in all_lexicons:
        for col_name, pattern in lexicon_group.items():
            df[col_name] = text_col.str.count(pattern, flags=re.IGNORECASE)
    
    print("...Finished counting occurrences.")

    # 5. Create Summary Features
    print("Creating summary boolean and categorical features...")

    # --- Combine counts into summary columns ---
    df['total_inclusive_lang'] = df[lexicon_gender_inclusive.keys()].sum(axis=1)
    df['total_exclusive_lang'] = df[lexicon_gender_exclusive.keys()].sum(axis=1)
    df['total_trans_identity'] = df[lexicon_gender_identity.keys()].sum(axis=1)
    
    df['total_traditional_prod'] = df[lexicon_product_traditional.keys()].sum(axis=1)
    df['total_alternative_prod'] = df[lexicon_product_alternative.keys()].sum(axis=1)
    
    df['total_product_stigma'] = df[lexicon_narrative_product_stigma.keys()].sum(axis=1)
    df['total_product_positive'] = df[lexicon_narrative_product_positive.keys()].sum(axis=1)
    df['total_identity_stigma'] = df[lexicon_narrative_identity_stigma.keys()].sum(axis=1)
    
    df['total_sustainability'] = df[lexicon_narrative_sustainability.keys()].sum(axis=1)
    df['total_health'] = df[lexicon_narrative_health.keys()].sum(axis=1)
    df['total_cost'] = df[lexicon_narrative_cost.keys()].sum(axis=1)

    # --- Create boolean (1/0) flags ---
    df['mentions_inclusive_lang'] = (df['total_inclusive_lang'] > 0).astype(int)
    df['mentions_exclusive_lang'] = (df['total_exclusive_lang'] > 0).astype(int)
    df['mentions_trans_identity'] = (df['total_trans_identity'] > 0).astype(int)
    
    df['mentions_traditional_prod'] = (df['total_traditional_prod'] > 0).astype(int)
    df['mentions_alternative_prod'] = (df['total_alternative_prod'] > 0).astype(int)
    
    df['mentions_product_stigma'] = (df['total_product_stigma'] > 0).astype(int)
    df['mentions_product_positive'] = (df['total_product_positive'] > 0).astype(int)
    df['mentions_identity_stigma'] = (df['total_identity_stigma'] > 0).astype(int)
    df['mentions_sustainability'] = (df['total_sustainability'] > 0).astype(int)
    df['mentions_health'] = (df['total_health'] > 0).astype(int)
    df['mentions_cost'] = (df['total_cost'] > 0).astype(int)

    # --- Create powerful categorical columns for analysis ---
    
    # Language Type
    def categorize_language(row):
        incl = row['mentions_inclusive_lang']
        excl = row['mentions_exclusive_lang']
        if incl == 1 and excl == 0:
            return 'Inclusive Only'
        if incl == 0 and excl == 1:
            return 'Exclusive Only'
        if incl == 1 and excl == 1:
            return 'Both'
        return 'None'
    
    df['language_category'] = df.apply(categorize_language, axis=1)

    # Product Type
    def categorize_product(row):
        trad = row['mentions_traditional_prod']
        alt = row['mentions_alternative_prod']
        if trad == 1 and alt == 0:
            return 'Traditional Only'
        if trad == 0 and alt == 1:
            return 'Alternative Only'
        if trad == 1 and alt == 1:
            return 'Both'
        return 'None'

    df['product_category'] = df.apply(categorize_product, axis=1)
    
    print("...Finished creating summary features.")
    
    # 6. Save the new feature-engineered DataFrame
    df.to_csv('MASTER_feature_engineered.csv', index=False)
    print("\n--- Phase 2 Complete. ---")
    print("Saved 'MASTER_feature_engineered.csv' for our next phase.")
    print(f"This new file contains all original data + {len(df.columns) - 14} new feature columns.") # 14 original cols

    # 7. Print analysis of new features
    print("\n--- Analysis of New Features ---")
    print("\nLanguage Category Distribution:")
    print(df['language_category'].value_counts(normalize=True))
    
    print("\nProduct Category Distribution:")
    print(df['product_category'].value_counts(normalize=True))
    
    print("\nNarrative Mentions (as % of all documents):")
    print(f"Trans Identity: {df['mentions_trans_identity'].mean():.2%}")
    print(f"Product Stigma/Fear: {df['mentions_product_stigma'].mean():.2%}")
    print(f"Product Positives: {df['mentions_product_positive'].mean():.2%}")
    print(f"Identity Stigma (Dysphoria/Exclusion): {df['mentions_identity_stigma'].mean():.2%}")
    print(f"Sustainability: {df['mentions_sustainability'].mean():.2%}")
    print(f"Health: {df['mentions_health'].mean():.2%}")
    print(f"Cost: {df['mentions_cost'].mean():.2%}")
    
    print("\nBasic stats on total counts (gives a sense of noise level):")
    print(df[['total_inclusive_lang', 'total_exclusive_lang', 'total_trans_identity', 'total_traditional_prod', 'total_alternative_prod', 'total_product_stigma', 'total_identity_stigma']].describe())

except FileNotFoundError:
    print(f"Error: Could not find 'MASTER_combined_and_cleaned.csv'.")
    print("Please make sure the output from Phase 1 is in the same directory.")
except Exception as e:
    print(f"An error occurred: {e}")

--- Phase 2 (REVISED): Text Preprocessing and Enhanced Feature Engineering ---
Successfully loaded 'MASTER_combined_and_cleaned.csv'. Shape: (100875, 14)
Prepared 'text_to_analyze' column (lowercase, fillna).

Starting feature engineering (counting ENHANCED lexicon occurrences)...
...Finished counting occurrences.
Creating summary boolean and categorical features...
...Finished creating summary features.

--- Phase 2 Complete. ---
Saved 'MASTER_feature_engineered.csv' for our next phase.
This new file contains all original data + 66 new feature columns.

--- Analysis of New Features ---

Language Category Distribution:
language_category
None              0.904971
Exclusive Only    0.094047
Both              0.000634
Inclusive Only    0.000347
Name: proportion, dtype: float64

Product Category Distribution:
product_category
None                0.785537
Traditional Only    0.142444
Both                0.039901
Alternative Only    0.032119
Name: proportion, dtype: float64

Narrative Menti